# Multiple positions on the IX83 unit 2

The same timelapse as `01_simple_acquisition`, at several fields of view.
You pick the positions in the MDA widget of napari-micromanager, and the
notebook reads them from there with `utils.generate_fov_positions`. Every
frame is segmented, tracked and measured per position. There is no
stimulation.

## 1. Connect to the IX83 unit 2

`IX83Unit2()` loads `IX83Unit2.cfg`, applies the `System/Startup` preset
(opens the epi shutters, halogen off), sets the `Binning` preset `1x1` and
selects the `Channel` group. Pass `config=r"..."` to load another file.
Close MetaMorph and any other Micro-Manager first: a device can only be
opened by one program at a time.

In [ ]:
from faro_aoki.ix83_unit2 import IX83Unit2
import faro.core.utils as utils

mic = IX83Unit2()
utils.print_configs(mic.mmc)

## 2. Open napari

napari-micromanager runs on the core of `mic`. Do not run
`faro-napari ix83unit2` at the same time: it opens the hardware a second time.

`open_napari(mic)` sets napari-micromanager up as `faro-napari ix83unit2` does:
the widgets handle the core's events on the GUI thread (faro acquires on
its own thread), and the Illumination Control dock and the Stage Map
button of `IX83Unit2` are added. It also lets `getImage()` read a snapped
frame more than once: the PVCAM camera hands out each frame only once,
and plain napari-micromanager reads it twice ("Camera image buffer read
failed"). Do not build `MainWindow(viewer, mmcore=...)`
by hand: its MDA widget then stops faro's run before the first frame.

In [ ]:
from faro.napari_launcher import open_napari

viewer, mm_widget = open_napari(mic)

## 3. Pick the positions in the MDA widget

1. Click **MDA** in the napari-micromanager toolbar. The MDA widget opens as
   a dock.
2. Tick **Positions** in the MDA widget.
3. For each field: move the stage there (Live view and the stage controls
   help), then click **Add** in the position table.

Only the position table is read; the channels, the time plan and the
**Run** button of the MDA widget are not used. faro runs the acquisition
below.

**Focus:** a position with a `z` value makes faro move the focus drive to
it on every visit. To keep the focus where the ZDC holds it, leave `z` out
of the positions (untick the Z column of the position table). The table in
step 5 shows which positions carry a `z`.

## 4. Settings

`power` is the level of the Spectra LED of the preset (0-99, see
`IX83Unit2.POWER_PROPERTIES`). Give every fluorescence channel a power:
the presets store none, so without it the LED keeps its last level.
The powers below are starting points; tune them on your sample.

`CHANNEL` must be a preset of the `Channel` group. `SEGMENTER` picks
`"cellpose"` (needs `uv sync --group cellpose` and a GPU) or `"otsu"`.
`INTERVAL_S` must leave time to visit every position: when they do not
all fit, `apply_fov_batching` in step 7 splits them into batches.
`WRITER` is `"omezarr"` (one store) or `"tiff"` (one file per frame).
Every run gets its own folder in `data/` next to this notebook.

In [ ]:
import os
from datetime import datetime

from faro.core.data_structures import PowerChannel

CHANNEL = PowerChannel(  # nuclear marker, Spectra red LED
    config="iRFP", exposure=100, group=mic.CHANNEL_GROUP, power=50
)
INTERVAL_S = 10.0  # seconds between frames of one position
N_FRAMES = 10
TIME_PER_FOV_S = 2.0  # stage move + acquisition per position, an estimate
SEGMENTER = "cellpose"  # or "otsu"
WRITER = "omezarr"  # or "tiff"

path = os.path.abspath(
    os.path.join("data", f"multi-position-{datetime.now():%Y%m%d-%H%M%S}")
)  # data/ next to this notebook

## 5. Read the positions from the MDA widget

`generate_fov_positions(mic, viewer=viewer)` reads the position table of
the MDA dock. Run this cell again after you change the table.

In [ ]:
import pandas as pd
import faro.core.utils as utils

fov_positions = utils.generate_fov_positions(mic, viewer=viewer)
assert fov_positions, "no positions: add them in the MDA widget (step 3)"

pd.DataFrame(fov_positions)

## 6. Set up the processing pipeline

Cellpose (or Otsu thresholding) for segmentation, trackpy for tracking
and `SimpleFE` for area and intensity per cell. Tracking runs per
position. Cellpose without a CUDA GPU falls back to the CPU, see
`01_simple_acquisition`.

In [ ]:
from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.simple import SimpleFE
from faro.tracking.trackpy import TrackerTrackpy

if SEGMENTER == "cellpose":
    import torch
    from faro.segmentation.cellpose_v4 import CellposeV4

    if torch.cuda.is_available():
        print(
            f"Cellpose on {torch.cuda.get_device_name(0)} (torch {torch.__version__})"
        )
    else:
        print(
            f"WARNING: torch {torch.__version__} sees no CUDA GPU, Cellpose runs on "
            "the CPU. Update the NVIDIA driver or set SEGMENTER = 'otsu'."
        )
    segmentator = CellposeV4()
else:
    from faro.segmentation.base import OtsuSegmentator

    segmentator = OtsuSegmentator()

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=segmentator,
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
)

## 7. Define the experiment

One `RTMSequence` over all positions. Every timepoint visits the positions
in the order of the table. `apply_fov_batching` checks whether
`len(positions) * TIME_PER_FOV_S` fits into `INTERVAL_S`, and splits the
positions into batches with their own timing when it does not.

In [ ]:
from faro.core.data_structures import RTMSequence

seq = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=fov_positions,
    channels=[CHANNEL],
)
events = utils.apply_fov_batching(list(seq), time_per_fov=TIME_PER_FOV_S)
print(f"{len(events)} events at {len(fov_positions)} positions")
utils.events_to_dataframe(events).sort_values("timestep").head(2 * len(fov_positions))

## 8. Run the experiment

`validate_events` checks the channel names and exposures against the
hardware before anything moves. `run_experiment` returns at once: the
acquisition and the pipeline run in the background, and napari stays
responsive and shows the frames. The experiment status dock opens with
the run.

In [ ]:
from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter, TiffWriter
from faro.widgets import ExperimentStatusWidget

if WRITER == "omezarr":
    writer = OmeZarrWriter(storage_path=path)
else:
    writer = TiffWriter(storage_path=path)

ctrl = Controller(mic, pipeline, writer=writer)
assert ctrl.validate_events(events), "fix the warnings above before running"
# Experiment status (progress, lag, errors), docked when the run starts.
viewer.window.add_dock_widget(
    ExperimentStatusWidget(ctrl), name="experiment status", area="right"
)
handle = ctrl.run_experiment(events)

`finish_experiment()` waits for the run, drains the pipeline and closes
the writer. napari does not update while this cell runs, so run it when
the status dock shows the run is done.

In [ ]:
handle.wait()
ctrl.finish_experiment()
print("results in", path)

## 9. Inspect the results

One parquet file of tracks per position, `tracks/<position>_latest.parquet`.
With several positions the OME-Zarr raw array is `(t, p, c, y, x)`; TIFF
files are named `<position>_<frame>.tiff`.

In [ ]:
for fov in range(len(fov_positions)):
    tracks = pd.read_parquet(os.path.join(path, "tracks", f"{fov}_latest.parquet"))
    print(
        f"Position {fov}: {tracks['particle'].nunique()} cells "
        f"across {tracks['timestep'].nunique()} frames"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import zarr

n_pos, t_last = len(fov_positions), N_FRAMES - 1


def last_frame(fov):
    """First channel of the last frame of one position."""
    if WRITER == "omezarr":
        raw = zarr.open(os.path.join(path, "acquisition.ome.zarr"), mode="r")["0"]
        frame = raw[t_last, fov] if raw.ndim == 5 else raw[t_last]
    else:
        name = f"{fov:03d}_{t_last:05d}.tiff"
        frame = tifffile.imread(os.path.join(path, "raw", name))
    frame = np.asarray(frame)
    return frame if frame.ndim == 2 else frame[0]


fig, axes = plt.subplots(1, n_pos, figsize=(4 * n_pos, 4), squeeze=False)
for fov, ax in enumerate(axes[0]):
    ax.imshow(last_frame(fov), cmap="gray")
    ax.set_title(f"Position {fov} (last frame)")
    ax.axis("off")
plt.tight_layout()